<a href="https://colab.research.google.com/github/ardha27/AI-Song-Cover-RVC/blob/main/Hina_Mod_AICoverGen_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# AICoverGen CLI (No WebUI - Anti Ban Colab)
### Fixed version for Google Colab 2025+ (Python 3.12, NumPy 2.x, PyTorch 2.10)

**Cara pakai:**
1. Jalankan **Cell 1** (Downgrade NumPy) lalu runtime akan restart otomatis
2. Setelah restart, jalankan **Cell 2 sampai Cell 6 (Download Model)** secara berurutan
3. Di Cell 7 (Generate AI Cover), masukkan link YouTube dan nama model RVC Anda lalu jalankan untuk membuat cover tanpa WebUI.

**Keuntungan CLI:** Karena tidak pakai Gradio/WebUI, Google Colab tidak akan mendeteksi dan memblokir runtime Anda!

In [ ]:
#@title Step 0: Downgrade NumPy (JALANKAN INI PERTAMA, runtime akan restart)
#@markdown NumPy 2.x tidak kompatibel dengan fairseq yang dibutuhkan RVC.
#@markdown Cell ini akan downgrade NumPy ke 1.26.4 lalu restart runtime otomatis.
#@markdown **Setelah restart, JANGAN jalankan cell ini lagi. Langsung lanjut ke cell berikutnya.**

import subprocess, sys, os

# Check current numpy version
try:
    import numpy as np
    current_version = np.__version__
    print(f"Current NumPy version: {current_version}")
    
    if int(current_version.split('.')[0]) >= 2:
        print("NumPy 2.x detected. Downgrading to 1.26.4 for fairseq compatibility...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'numpy==1.26.4', '-q'])
        print("NumPy downgraded. Restarting runtime...")
        print("")
        print("Setelah restart, JANGAN jalankan cell ini lagi.")
        print("Langsung jalankan cell 'Clone repository' ke bawah.")
        os.kill(os.getpid(), 9)  # Force restart runtime
    else:
        print(f"NumPy {current_version} is already compatible. Proceed to next cell.")
except Exception as e:
    print(f"Error: {e}")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'numpy==1.26.4', '-q'])
    os.kill(os.getpid(), 9)

In [ ]:
#@title Clone repository
from IPython.display import clear_output, Javascript
import codecs
import threading
import time
import os
import numpy as np
print(f"NumPy version: {np.__version__} (should be 1.26.x)")

#=======================Auto Edit======================

#@markdown ---
#@markdown Switch between ```-1 0 1``` or ```-12 0 12``` pitch change control

#@markdown This can only be changed once, you need to restart the whole thing if you wanna change it again
Pitch_Change="12" #@param ['1','12']

#@markdown Enable if you want to install the Program to your Drive
if Pitch_Change=="1":
  cloneing=codecs.decode('uggcf://tvguho.pbz/FbpvnyylVarcgJrro/NVPbireTra.tvg','rot_13')
else:
  cloneing=codecs.decode('uggcf://tvguho.pbz/nequn27/NVPbireTra-Zbq.tvg','rot_13')
#=====================Auto Edit End================
Install_To_Drive=True #@param {type:"boolean"}


#====================Use Drive============
if Install_To_Drive == True:
    from google.colab import drive
    drive.mount('/content/drive')
    repo_path = '/content/drive/MyDrive/Hina_RVC'
else:
    repo_path = '/content/Hina_RVC'

if os.path.isdir(repo_path):
    print(f"Repository already exists at {repo_path}, pulling the latest changes.")
    %cd $repo_path
    !git pull
else:
    print(f"Cloning repository to {repo_path}")
    !git clone $cloneing Hina_RVC
    if Install_To_Drive == True:
        !mv Hina_RVC /content/drive/MyDrive/
        %cd /content/drive/MyDrive/Hina_RVC
    else:
        %cd Hina_RVC

!rm -rf sample_data


def update_timer_and_print():
    global timer
    while True:
        hours, remainder = divmod(timer, 3600)
        minutes, seconds = divmod(remainder, 60)
        timer_str = f'{hours:02}:{minutes:02}:{seconds:02}'
        print(f'\rTimer: {timer_str}', end='', flush=True)
        time.sleep(1)
        timer += 1
timer = 0
threading.Thread(target=update_timer_and_print, daemon=True).start()



clear_output()
print("Done Cloning Repository")

In [ ]:
#@title Install requirements (FIXED for Colab 2025+ / Python 3.12)
from IPython.display import clear_output
import subprocess, sys

# ============================================================
# Step 1: Pre-install build dependencies (Downgrade pip)
# ============================================================
print("[1/7] Installing build dependencies...")
# pip >= 24.1 breaks omegaconf metadata parsing (which fairseq needs)
# We MUST downgrade pip to < 24.1 as recommended by the error log
!pip install -q "pip<24.1" setuptools wheel Cython

# Ensure numpy stays at 1.26.x
!pip install -q "numpy==1.26.4"

# ============================================================
# Step 2: Clean up requirements.txt
# ============================================================
print("[2/7] Cleaning up requirements.txt...")

# Remove packages that conflict or need special handling
!sed -i '/torch==/d' requirements.txt
!sed -i '/torchaudio==/d' requirements.txt
!sed -i '/numpy/d' requirements.txt
!sed -i '/librosa/d' requirements.txt
!sed -i '/Requests==/d' requirements.txt
!sed -i '/scipy/d' requirements.txt
!sed -i '/soundfile/d' requirements.txt
!sed -i '/tqdm==/d' requirements.txt
!sed -i '/onnxruntime/d' requirements.txt
!sed -i '/fairseq/d' requirements.txt
!sed -i '/pyworld/d' requirements.txt
!sed -i '/gradio/d' requirements.txt
!sed -i '/faiss-cpu/d' requirements.txt
!sed -i '/pedalboard/d' requirements.txt
!sed -i '/--find-links/d' requirements.txt

# ============================================================
# Step 3: Install cleaned requirements
# ============================================================
print("[3/7] Installing cleaned requirements...")
!pip install -q -r requirements.txt

# ============================================================
# Step 4: Install fairseq (the most problematic package)
# ============================================================
print("[4/7] Installing fairseq...")
# Install omegaconf and hydra first (fairseq dependencies)
!pip install -q omegaconf hydra-core
# Use fairseq-fixed for Python 3.12 compatibility (bypasses distutils removal)
!pip install -q fairseq-fixed

# ============================================================
# Step 5: Install pyworld
# ============================================================
print("[5/7] Installing pyworld...")
!pip install -q pyworld --no-build-isolation

# ============================================================
# Step 6: Install remaining packages
# ============================================================
print("[6/7] Installing remaining packages...")
# Install ONNX Runtime GPU compatible with CUDA 12 (1.19.2)
!pip install -q onnxruntime-gpu==1.19.2
# Gunakan versi gradio terbaru untuk menghindari error HuggingFace HfFolder di versi lama
!pip install -q "gradio>=4.0.0"
!pip install -q librosa soundfile "scipy>=1.11.0,<2.0.0"
!pip install -q sox yt-dlp gdown faiss-cpu pedalboard

# Re-pin numpy in case something upgraded it
!pip install -q "numpy==1.26.4"

# ============================================================
# Step 7: Patch torch.load for fairseq Dictionary safe globals
# ============================================================
print("[7/7] Applying patches...")
try:
    import torch
    from fairseq.data.dictionary import Dictionary
    if hasattr(torch.serialization, 'add_safe_globals'):
        torch.serialization.add_safe_globals([Dictionary])
        print("    PyTorch safe globals patch applied successfully.")
except:
    pass

# Patch webui.py to remove deprecated 'show_share_button' argument that crashes new Gradio
!sed -i 's/, show_share_button=False//g' src/webui.py
!sed -i 's/, show_share_button=True//g' src/webui.py
!sed -i 's/show_share_button=False, //g' src/webui.py
!sed -i 's/show_share_button=True, //g' src/webui.py
print("    WebUI Gradio patch applied successfully.")

clear_output()
print("Finished Installing Requirements")

# Install system-level sox binary
!sudo apt update -qq > /dev/null 2>&1
!sudo apt install -y -qq sox libsox-fmt-all > /dev/null 2>&1

clear_output()

# Final verification
import numpy as np
import torch
print("All requirements installed successfully!")
print(f"   NumPy:   {np.__version__}")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA:    {torch.cuda.is_available()} ({'GPU: ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'})")
try:
    import fairseq
    print(f"   Fairseq: {fairseq.__version__}")
except:
    print(f"   Fairseq: installed (version unavailable)")
try:
    import sox
    print(f"   Sox:     installed")
except:
    print(f"   Sox:     WARNING - not found!")
print("")
print("Proceed to the next cell")

In [ ]:
#@title Download MDXNet Vocal Separation and Hubert Base Models
import codecs

# Apply safe globals patch before downloading models
import torch
try:
    from fairseq.data.dictionary import Dictionary
    if hasattr(torch.serialization, 'add_safe_globals'):
        torch.serialization.add_safe_globals([Dictionary])
except:
    pass

models=codecs.decode('fep/qbjaybnq_zbqryf.cl','rot_13')
!python $models
from IPython.display import clear_output
clear_output()
print("Finished Downloading Voice Separation Model and Hubert Base Model")

In [ ]:
#@title Download RVC Model (Opsional)
#@markdown Jalankan cell ini JIKA Anda belum punya modelnya. Masukkan link download (format .zip) langsung (misal dari HuggingFace) dan nama model.
MODEL_LINK = "" #@param {type:"string"}
MODEL_NAME = "" #@param {type:"string"}

import os
import urllib.request
import zipfile
import shutil

model_dir = os.path.join("rvc_models", MODEL_NAME)
os.makedirs(model_dir, exist_ok=True)
zip_path = f"{MODEL_NAME}.zip"

print(f"Downloading {MODEL_NAME} dari {MODEL_LINK}...")
try:
    urllib.request.urlretrieve(MODEL_LINK, zip_path)
    print("Extracting...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(model_dir)
    
    # Pindahkan file .pth dan .index ke luar (jika di dalam subfolder)
    for root, dirs, files in os.walk(model_dir):
        for file in files:
            if file.endswith(".pth") or file.endswith(".index"):
                source = os.path.join(root, file)
                target = os.path.join(model_dir, file)
                if source != target:
                    shutil.move(source, target)
    
    os.remove(zip_path)
    print(f"\nSUKSES! Model {MODEL_NAME} sudah siap digunakan.")
    print(f"Pastikan RVC_DIRNAME di cell bawah diisi dengan: {MODEL_NAME}")
except Exception as e:
    print(f"ERROR: {e}\nPastikan link download valid dan langsung mengarah ke file .zip")

In [ ]:
#@title Generate AI Cover (Full CLI Mode)
#@markdown Pastikan model RVC Anda sudah diupload ke folder `Hina_RVC/rvc_models/nama_folder_model` (yang berisi file .pth dan .index)

#@markdown ### 🎵 Pengaturan Utama
SONG_INPUT = "" #@param {type:"string"}
RVC_DIRNAME = "" #@param {type:"string"}
PITCH_CHANGE = 0 #@param {type:"integer"}
PITCH_CHANGE_ALL = 0 #@param {type:"integer"}

#@markdown ### 🗣️ Pengaturan Suara (Voice & Audio)
INDEX_RATE = 0.5 #@param {type:"slider", min:0, max:1, step:0.01}
FILTER_RADIUS = 3 #@param {type:"slider", min:0, max:7, step:1}
RMS_MIX_RATE = 0.25 #@param {type:"slider", min:0, max:1, step:0.01}
PROTECT = 0.33 #@param {type:"slider", min:0, max:0.5, step:0.01}
PITCH_DETECTION_ALGO = "rmvpe" #@param ["rmvpe", "mangio-crepe"]
CREPE_HOP_LENGTH = 128 #@param {type:"integer"}

#@markdown ### 🎚️ Pengaturan Volume (Decibels)
MAIN_VOCALS_VOLUME_CHANGE = 0 #@param {type:"integer"}
BACKUP_VOCALS_VOLUME_CHANGE = 0 #@param {type:"integer"}
INSTRUMENTAL_VOLUME_CHANGE = 0 #@param {type:"integer"}

#@markdown ### 🎸 Pengaturan Reverb (Efek Ruangan)
REVERB_SIZE = 0.15 #@param {type:"slider", min:0, max:1, step:0.01}
REVERB_WETNESS = 0.2 #@param {type:"slider", min:0, max:1, step:0.01}
REVERB_DRYNESS = 0.8 #@param {type:"slider", min:0, max:1, step:0.01}
REVERB_DAMPING = 0.7 #@param {type:"slider", min:0, max:1, step:0.01}

#@markdown ### 💾 Output
OUTPUT_FORMAT = "mp3" #@param ["mp3", "wav"]
KEEP_FILES = False #@param {type:"boolean"}

keep_arg = "-k" if KEEP_FILES else ""

command = f'!python src/main.py -i "{SONG_INPUT}" -dir "{RVC_DIRNAME}" -p {PITCH_CHANGE} {keep_arg} -ir {INDEX_RATE} -fr {FILTER_RADIUS} -rms {RMS_MIX_RATE} -palgo {PITCH_DETECTION_ALGO} -hop {CREPE_HOP_LENGTH} -pro {PROTECT} -mv {MAIN_VOCALS_VOLUME_CHANGE} -bv {BACKUP_VOCALS_VOLUME_CHANGE} -iv {INSTRUMENTAL_VOLUME_CHANGE} -pall {PITCH_CHANGE_ALL} -rsize {REVERB_SIZE} -rwet {REVERB_WETNESS} -rdry {REVERB_DRYNESS} -rdamp {REVERB_DAMPING} -oformat {OUTPUT_FORMAT}'

# Jalankan perintah menggunakan API IPython
get_ipython().system(command.replace('!', ''))

print("\nSelesai! Hasil audio ada di folder 'song_output' di dalam direktori Hina_RVC")

[![](https://i.pinimg.com/474x/de/72/9e/de729ecfa41b69901c42c82fff752414.jpg)](https://discordlookup.com/user/444684887363026974)